# Deploy Gateway & Interceptor

Deploy the AgentCore Gateway and Interceptor Lambda.

## Prerequisites

- ✅ Run `03-deploy-mcp-server.ipynb` first
- ✅ MCP Server Runtime ARN saved to SSM

## What This Notebook Does

1. Deploys Gateway Interceptor Lambda
2. Creates AgentCore Gateway
3. Configures OAuth token validation
4. Saves Gateway ARN to SSM

## Next Notebook

- **05-deploy-agent.ipynb**

In [3]:
import sys
sys.path.insert(0, '.')  # Add current directory to path
from pathlib import Path
from aws_session_utils import get_aws_session

# Get validated AWS session with SSO support
session, AWS_REGION, AWS_ACCOUNT_ID = get_aws_session()

# Initialize AWS clients
lambda_client = session.client('lambda', region_name=AWS_REGION)
ssm_client = session.client('ssm', region_name=AWS_REGION)

print('✅ Setup complete')
print(f'   Region: {AWS_REGION}')
print(f'   Account ID: {AWS_ACCOUNT_ID}')

🔍 Using default AWS credentials (no profile specified)
⚠️  No AWS region configured, using default: us-east-1
   To set your region:
   - Environment variable: export AWS_DEFAULT_REGION=your-region
   - AWS CLI: aws configure set region your-region

✅ AWS Credentials Validated
   Region: us-east-1
   Account ID: XXXXXXXXXXXX
   Profile: default
   Auth method: AWS SSO

✅ Setup complete
   Region: us-east-1
   Account ID: XXXXXXXXXXXX


## Step 1: Deploy Interceptor Lambda

In [4]:
import subprocess

# Run deploy_interceptor.py to deploy Lambda function
result = subprocess.run(
    ['python', 'deploy_interceptor.py'],
    cwd='gateway-setup/interceptor',
    capture_output=True,
    text=True
)

print(result.stdout)
if result.returncode != 0:
    print('❌ Error:', result.stderr)
else:
    print('\n✅ Interceptor Lambda deployed!')

Gateway Interceptor Lambda Deployment

🔍 Initializing AWS session...
🔍 Using default AWS credentials (no profile specified)
⚠️  No AWS region configured, using default: us-east-1
   To set your region:
   - Environment variable: export AWS_DEFAULT_REGION=your-region
   - AWS CLI: aws configure set region your-region

✅ AWS Credentials Validated
   Region: us-east-1
   Account ID: XXXXXXXXXXXX
   Profile: default
   Auth method: AWS SSO


🔍 Loading configuration from SSM Parameter Store...
✅ Configuration loaded from SSM
   Cognito User Pool ID: us-east-1_3jSGececJ
   Cognito App Client ID: comfh80kmr0a3mlrusl6cmulu

Step 1: Packaging Lambda Function

📦 Packaging Lambda function...
✅ Package created: interceptor-lambda.zip

Step 2: Creating Lambda Execution Role

🔑 Creating Lambda execution role...
ℹ️  Role InsuranceClaimsGatewayInterceptorRole already exists, retrieving ARN
   ✅ Role policy updated
✅ Lambda role ready: arn:aws:iam::XXXXXXXXXXXX:role/InsuranceClaimsGatewayInterceptorRol

## Step 2: Get Required ARNs

In [5]:
# Get Interceptor Lambda ARN from SSM (saved by deploy_interceptor.py)
INTERCEPTOR_ARN = ssm_client.get_parameter(
    Name='/app/lakehouse-agent/interceptor-lambda-arn'
)['Parameter']['Value']
print(f'✅ Interceptor ARN: {INTERCEPTOR_ARN}')

# Get MCP Server Runtime ARN from SSM
MCP_SERVER_RUNTIME_ARN = ssm_client.get_parameter(
    Name='/app/lakehouse-agent/mcp-server-runtime-arn'
)['Parameter']['Value']
print(f'✅ MCP Server ARN: {MCP_SERVER_RUNTIME_ARN}')

# Get Cognito User Pool ARN
COGNITO_USER_POOL_ID = ssm_client.get_parameter(
    Name='/app/lakehouse-agent/cognito-user-pool-id'
)['Parameter']['Value']
COGNITO_USER_POOL_ARN = f'arn:aws:cognito-idp:{AWS_REGION}:{AWS_ACCOUNT_ID}:userpool/{COGNITO_USER_POOL_ID}'
print(f'✅ Cognito User Pool ARN: {COGNITO_USER_POOL_ARN}')

✅ Interceptor ARN: arn:aws:lambda:us-east-1:XXXXXXXXXXXX:function:lakehouse-gateway-interceptor
✅ MCP Server ARN: arn:aws:bedrock-agentcore:us-east-1:XXXXXXXXXXXX:runtime/lakehouse_mcp_server-iPIYzlC3zh
✅ Cognito User Pool ARN: arn:aws:cognito-idp:us-east-1:XXXXXXXXXXXX:userpool/us-east-1_3jSGececJ


## Step 3: Create AgentCore Gateway

This will create the gateway and automatically configure it with the MCP server and interceptor.

In [8]:
# Create AgentCore Gateway
result = subprocess.run([
    'python', 'create_gateway.py',
    '--yes'  # Auto-confirm for notebook execution
], cwd='gateway-setup', capture_output=True, text=True)

print(result.stdout)
if result.returncode != 0:
    print('❌ Error:', result.stderr)
else:
    print('\n✅ Gateway created!')
    print('\n📋 Gateway ARN saved to SSM Parameter Store')

AgentCore Gateway Setup
🔍 Using default AWS credentials (no profile specified)
⚠️  No AWS region configured, using default: us-east-1
   To set your region:
   - Environment variable: export AWS_DEFAULT_REGION=your-region
   - AWS CLI: aws configure set region your-region

✅ AWS Credentials Validated
   Region: us-east-1
   Account ID: XXXXXXXXXXXX
   Profile: default
   Auth method: AWS SSO

✅ Configuration loaded
   Region: us-east-1
   Account: XXXXXXXXXXXX

🔍 Loading configuration from SSM Parameter Store...
   ✅ M2M Client ID: 6dls2pick0ekciocqckh6s02ss
   ✅ M2M Client Secret: ****** (loaded)
   ✅ MCP Server Runtime ARN: arn:aws:bedrock-agentcore:us-east-1:XXXXXXXXXXXX:runtime/lakehouse_mcp_server-iPIYzlC3zh
   ✅ Interceptor Lambda ARN: arn:aws:lambda:us-east-1:XXXXXXXXXXXX:function:lakehouse-gateway-interceptor
   ✅ Cognito User Pool ARN: arn:aws:cognito-idp:us-east-1:XXXXXXXXXXXX:userpool/us-east-1_3jSGececJ
   ✅ Cognito App Client ID: comfh80kmr0a3mlrusl6cmulu
   ✅ Cognito Clie

## Step 4: Verify Gateway Configuration

The create_gateway.py script automatically saves the Gateway ARN to SSM.
Run this cell to verify the deployment.

In [9]:
# Verify Gateway configuration in SSM
print("Verifying Gateway configuration in SSM...\n")

parameters_to_check = [
    '/app/lakehouse-agent/gateway-arn',
    '/app/lakehouse-agent/gateway-id',
    '/app/lakehouse-agent/gateway-url',
]

all_found = True
for param_name in parameters_to_check:
    try:
        response = ssm_client.get_parameter(Name=param_name)
        value = response['Parameter']['Value']
        print(f'✅ {param_name}')
        print(f'   Value: {value}')
    except ssm_client.exceptions.ParameterNotFound:
        print(f'❌ {param_name} - NOT FOUND')
        all_found = False
    except Exception as e:
        print(f'⚠️  {param_name} - ERROR: {e}')
        all_found = False

if all_found:
    print('\n✅ Gateway configuration verified in SSM!')
else:
    print('\n⚠️  Gateway parameters missing.')
    print('    The create_gateway.py script should have saved these automatically.')
    print('    Check the deployment output for errors.')

Verifying Gateway configuration in SSM...

✅ /app/lakehouse-agent/gateway-arn
   Value: arn:aws:bedrock-agentcore:us-east-1:XXXXXXXXXXXX:gateway/lakehouse-gateway-1ekjecoowq
✅ /app/lakehouse-agent/gateway-id
   Value: lakehouse-gateway-1ekjecoowq
✅ /app/lakehouse-agent/gateway-url
   Value: https://lakehouse-gateway-1ekjecoowq.gateway.bedrock-agentcore.us-east-1.amazonaws.com/mcp

✅ Gateway configuration verified in SSM!


## Summary

✅ **Gateway & Interceptor Deployment Complete!**

**What was created:**
- Interceptor Lambda (JWT validation)
- AgentCore Gateway (routing)

All configuration saved to SSM Parameter Store.

**Next Steps:**
Run **05-deploy-agent.ipynb**